## Hugging Face Tutorial

### Env Checking

In [1]:
import os
import sys
import torch

print("=== Basic Environment Check ===")
print("Python executable:", sys.executable)
print("Current working directory:", os.getcwd())

print("\n=== PyTorch / CUDA Check ===")
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    x = torch.tensor([1.0, 2.0, 3.0]).cuda()
    print("GPU tensor device:", x.device)
else:
    print("⚠️ CUDA not detected in this notebook kernel")

=== Basic Environment Check ===
Python executable: c:\Users\Minkyu Ham\Desktop\stat359\.venv\Scripts\python.exe
Current working directory: c:\Users\Minkyu Ham\Desktop\stat359\student\hf_tutorial

=== PyTorch / CUDA Check ===
Torch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
GPU tensor device: cuda:0


### Part A: pipeline

In [11]:
from transformers import pipeline

In [12]:
device = 0 if torch.cuda.is_available() else -1

In [17]:
base_model_checkpoint = "distilbert-base-uncased"

base_classifier = pipeline(
    "sentiment-analysis",
    model=base_model_checkpoint,
    tokenizer=base_model_checkpoint,
    device=device
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


In [18]:
before_review = "The cast is great and there are a few strong scenes, but the story drags for too long."
base_output = base_classifier(before_review)
print(base_output)

[{'label': 'LABEL_1', 'score': 0.5172842144966125}]


In [19]:
print("Movie review:", before_review)
print("Base model output:", base_output[0])

Movie review: The cast is great and there are a few strong scenes, but the story drags for too long.
Base model output: {'label': 'LABEL_1', 'score': 0.5172842144966125}


### Part B: Dataset

In [27]:
from datasets import load_dataset
from datasets import load_dataset_builder

In [28]:
# data preparation
dataset_name = "rotten_tomatoes"
builder = load_dataset_builder(dataset_name)  # meta info
builder.download_and_prepare()  # real data
print(f"Dataset downloaded/prepared at: {builder.cache_dir}")

Dataset downloaded/prepared at: C:\Users\Minkyu Ham\.cache\huggingface\datasets/rotten_tomatoes/default/0.0.0/aa13bc287fa6fcab6daf52f0dfb9994269ffea28


In [29]:
# data load
raw_datasets = load_dataset("rotten_tomatoes")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [33]:
# split data
small_train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = raw_datasets["validation"].shuffle(seed=42).select(range(300))

print(small_train_dataset, small_eval_dataset)
print("Train subset size:", len(small_train_dataset))
print("Eval subset size:", len(small_eval_dataset))
print("Example:", small_train_dataset[0])

Dataset({
    features: ['text', 'label'],
    num_rows: 1000
}) Dataset({
    features: ['text', 'label'],
    num_rows: 300
})
Train subset size: 1000
Eval subset size: 300
Example: {'text': '. . . plays like somebody spliced random moments of a chris rock routine into what is otherwise a cliche-riddled but self-serious spy thriller .', 'label': 0}


### Part C: Tokenizer

In [35]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(base_model_checkpoint)
tokenizer

DistilBertTokenizerFast(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [36]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

In [40]:
tokenize_function({"text": "Tokenizing Function Test. How is it?", "label": 1})

{'input_ids': [101, 19204, 6026, 3853, 3231, 1012, 2129, 2003, 2009, 1029, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [42]:
tokenized_train_dataset = small_train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = small_eval_dataset.map(tokenize_function, batched=True)

tokenized_train_dataset, tokenized_eval_dataset

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

(Dataset({
     features: ['text', 'label', 'input_ids', 'attention_mask'],
     num_rows: 1000
 }),
 Dataset({
     features: ['text', 'label', 'input_ids', 'attention_mask'],
     num_rows: 300
 }))

In [44]:
sample_text = "Hello"
sample_ids = tokenizer(sample_text, add_special_tokens=True)["input_ids"]  # add_special_tokens=True: BERT always add [CLS] and [SEP]

print("Sample text:", sample_text)
print("Encoded IDs:", sample_ids)
print("Decoded:", tokenizer.decode(sample_ids))
print("You should see IDs similar to [101, 7592, 102] for this tokenizer.")

Sample text: Hello
Encoded IDs: [101, 7592, 102]
Decoded: [CLS] hello [SEP]
You should see IDs similar to [101, 7592, 102] for this tokenizer.


### Side note

- Fine-tuning is a form of transfer learning.

### Part D: Fine-Tuning

In [ ]:
import evaluate
import numpy as np
from transformers import (
    AutoModelForSequenceClassification,  # 텍스트 분류용 모델 자동 호출
    TrainingArguments,  # 학습 설정 configure (batch size, epoch, log, save, fp16, ...)
    Trainer,  # PyTorch 학습 루프 대타 API
    DataCollatorWithPadding,  # 동적 padding (batch 내 최대 길이에 맞춰 padding)
    EarlyStoppingCallback  # 조기 종료 if no validation 성능 개선
)

In [47]:
# label mapping
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

In [48]:
# load model
model = AutoModelForSequenceClassification.from_pretrained(
    base_model_checkpoint,
    num_labels=2,  # 출력 logits 크기: [batch_size, 2]  //  분류 문제 마지막엔 Linear layer가 붙는데 이 출력 차원을 num_label로 설정
    id2label=id2label,
    label2id=label2id
)
model

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [50]:
# Technique 1: Layer Freezing
num_frozen_layers = 4

# freeze embeddings
for param in model.distilbert.embeddings.parameters():
    param.requires_grad = False

# freeze first 4 transformer layers
for layer in model.distilbert.transformer.layer[:num_frozen_layers]:
    for param in layer.parameters():
        param.requires_grad = False

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [51]:
# Check whether freezing worked well  -> 22% trainable = 78% freezed
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)  # .numel(): tensor 안의 총 원소 개수 = 파라미터 개수 // 여기선 역전파 계산 가능한 것 개수
total = sum(p.numel() for p in model.parameters())  # 여기서 전체
print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")  # 비율

Trainable params: 14,767,874 / 66,955,010 (22.06%)


In [52]:
# Define Metric: Accuracy
metric = evaluate.load("accuracy")  # 내 경우 ICD 코드 분류 클래스 불균형 있을 수 있음. -> F1은 가중치로 소수 클래스도 같이 확인

def compute_metrics(eval_pred):
    logits, labels = eval_pred  # logits: 점수, not 확률
    predictions = np.argmax(logits, axis=-1)  # argmax: 두 클래스 중 큰 쪽
    return metric.compute(predictions=predictions, references=labels)  # predictions와 labels를 비교해서 정확도 계산

In [53]:
# Technique 2: Dynamic Padding w/ datacollator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)  # 배치 안의 가장 긴 문장 길이에 맞춰서 padding, not max(512)
data_collator

DataCollatorWithPadding(tokenizer=DistilBertTokenizerFast(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')

In [56]:
# Setup training rules
training_args = TrainingArguments(
    output_dir="./distilbert-rotten-tomatoes",  # 모델 체크포인트 저장 경로

    learning_rate=2e-5,  # BERT 계열 fine-tuning 기본값 보통 2e-5 ~ 5e-5

    # 배치/누적 gradient accumulation
    per_device_train_batch_size=8,  # OoM 나면 배치 사이즈부터 낮추기 e.g.) 4   // GPU 하나당 배치 크기
    per_device_eval_batch_size=16,  # backward가 없어서 train 보다 크게 가능
    gradient_accumulation_steps=2,  # 실효 배치 크기는 8 * 2 = 16 처럼 동작 (VRAM은 8만큼 쓰면서, 16처럼 업데이트 안정화) // 배치 8을 2번 누적

    num_train_epochs=3,  # 데이터 전체 3번 반복
    weight_decay=0.01,  # L2 정규화 = 과적합 방지

    # 로깅/평가/저장
    eval_strategy="epoch",  # epoch이 끝날 때마다 검증
    save_strategy="epoch",  # """"""""""""""""" 저장
    logging_steps=10,  # 진행상황 자주 보여줌   // 10 step 마다 로그 출력 (step = 한 번의 foward + backward + update)  // 배치 8 + train 1000 => epoch 당 125 step -> 한 epoch에 약 12번 로그 출력
    
    # mixed precision
    fp16=torch.cuda.is_available(),   # 메모리 절약 + 속도 증가 - 수치 불안정/오류면 False로 끄기 시도

    # 스케줄러/워밍업
    lr_scheduler_type="cosine",  # 코사인 곡선 형태로 lr 줄임
    warmup_ratio=0.1,  # 보통 warmup_steps는 정수로 step 수 -> 약간 이상하긴함. -> warmup_ratio가 소수점 비율  // 초반에 작은 학습률 -> 점점 올림 -> 학습 안정화
    
    # gradient clipping
    max_grad_norm=1.0,  # 폭주(gradient explosion) 방지  // gradient의 L2 norm이 1.0을 넘으면 1.0으로 강제 축소 = 업데이트 크기 제한

    # Best Model
    load_best_model_at_end=True,  # 학습 끝나면 최고 성능 모델 자동 로드
    metric_for_best_model="accuracy",  # 최고 성능의 기준 = accuracy
    greater_is_better=True,  # 좋아졌다의 판단 기준 - accuracy는 커지는게 좋은 것 vs loss는 작아지는 것이 좋은 것

    save_total_limit=2,  # 에폭마다 저장할 때 너무 많아지니까 최근 2개만 남기기
    report_to="none"  # 학습 로그를 어디로 보낼지 e.g.) Tensor Board, WandB, MLflow 등
)

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False

In [60]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,  # 배치 구성 + 동적 padding
    compute_metrics=compute_metrics,  # accuracy 계산
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # early stopping
)
trainer

In [61]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.626000,0.577654,0.740000
2,0.386200,0.458617,0.783333
3,0.361600,0.435803,0.800000


TrainOutput(global_step=189, training_loss=0.5062927669949002, metrics={'train_runtime': 38.0588, 'train_samples_per_second': 78.825, 'train_steps_per_second': 4.966, 'total_flos': 397402195968000.0, 'train_loss': 0.5062927669949002, 'epoch': 3.0})

In [62]:
trainer.evaluate()

{'eval_loss': 0.4358028173446655,
 'eval_accuracy': 0.8,
 'eval_runtime': 1.9511,
 'eval_samples_per_second': 153.762,
 'eval_steps_per_second': 9.738,
 'epoch': 3.0}

### opt tech: Parameter-Efficient Fine-Tuning (LoRA)

- Parameter-Efficient Fine-Tuning (PEFT) - 그 중 대표가 LoRA.
- 기존 weight는 freeze, 작은 저차원 행렬 두개를 추가해서 그 행렬만 학습 -> 기존 모델은 그대로 유지 및 작은 adapter만 학습
- 메모리 효율, 시간 효율, 저장 파일 적음

In [63]:
from peft import LoraConfig, TaskType, get_peft_model

In [66]:
# 기존 모델 로드
lora_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_checkpoint,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [67]:
# LoRA Configure
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # 작업 지정, SEQ_CLS = sequence classification
    r=8,  # LoRA의 핵심 파라미터 = rank  // 보통 4~16.  클수록 표현력 업, 메모리 업
    lora_alpha=16,  # scaling factor = 업데이트 크기 조절 (보통 r의 2배)
    lora_dropout=0.1,  # 과적합 방지
    target_modules=["q_lin", "v_lin"],  # Transformer의 attention의 Q, K, V 중 보통 query와 value에 LoRA 적용
)
lora_config

LoraConfig(task_type=<TaskType.SEQ_CLS: 'SEQ_CLS'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'v_lin', 'q_lin'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)

In [70]:
lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()

# If you want to train LoRA adapters instead of the full model,
# pass `lora_model` to Trainer(model=...) and keep the same training pipeline.
# => 만약 쓸거면 Trainer()에서 model = lora_model로만 바꾸면 됨. 나머진 전부 동일.

trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


#### 결국 그래서 순서가 어떻게 되냐면..

1) 데이터/토크나이저 준비 (Part B/C)
```{python}
# 이전과 동일
```
2) base 분류 모델 로드
```{python}
from transformers import AutoModelForSequenceClassification

base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_checkpoint,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)
```
3) LoRA 설정 만들기
```{python}
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
)
```
4) base_model에 LoRA를 “삽입”해서 lora_model 생성
```{python}
from peft import get_peft_model

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()
```
5) TrainingArguments / compute_metrics 준비
```{python}
# 이전과 동일
```
6) Trainer 생성 시 model로 lora_model 넣기
```{python}
from transformers import Trainer

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    processing_class=tokenizer,   # 버전에 따라 tokenizer=tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
```
7) 학습/평가 실행
```{python}
trainer.train()
trainer.evaluate()
```

### Part E: Evaluation and Inference

In [71]:
eval_results = trainer.evaluate()
print("Evaluation metrics:", eval_results)

Evaluation metrics: {'eval_loss': 0.4358028173446655, 'eval_accuracy': 0.8, 'eval_runtime': 2.046, 'eval_samples_per_second': 146.628, 'eval_steps_per_second': 9.286, 'epoch': 3.0}


In [72]:
# 로컬 저장 - 같은 폴더에 저장하면 나중에 pipeline(model=..., tokenizer=...)로 바로 로드 가능
trainer.save_model("./my_fine_tuned_model")  # 모델 가중치 + config 저장
tokenizer.save_pretrained("./my_fine_tuned_model")  # tokenizer 파일(vocab 등) 저장

('./my_fine_tuned_model\\tokenizer_config.json',
 './my_fine_tuned_model\\special_tokens_map.json',
 './my_fine_tuned_model\\vocab.txt',
 './my_fine_tuned_model\\added_tokens.json',
 './my_fine_tuned_model\\tokenizer.json')

In [73]:
# 저장한 모델로 pipeline 생성
fine_tuned_classifier = pipeline(
    "sentiment-analysis",
    model="./my_fine_tuned_model",
    tokenizer="./my_fine_tuned_model",  # 원래 hf HUB 모델 이름이었는데 대신 로컬 경로 넣은 것
    device=device
)
fine_tuned_classifier

Device set to use cuda:0


In [74]:
# 같은 문장으로 before vs after
after_output = fine_tuned_classifier(before_review)[0]

print("Movie review:", before_review)
print("Before fine-tuning:", base_output)  # 레이블 보기 안좋음 + 점수 낮음
print("After fine-tuning:", after_output)

Movie review: The cast is great and there are a few strong scenes, but the story drags for too long.
Before fine-tuning: [{'label': 'LABEL_1', 'score': 0.5172842144966125}]
After fine-tuning: {'label': 'NEGATIVE', 'score': 0.8581476807594299}


In [ ]:
# FIN.